# Data Description
- Background: 兩位同名歷史連結再核對、四位G6正值及線上搜尋紀錄。
- date received: 20261108（本輪實際存取）
- Path to data file: data/raw/identity_recheck_20261108/；各原始網址見manifest與閱讀紀錄。
- Unit of observation: 一份HTTP回應、文字擷取或人工閱讀／搜尋紀錄。
- Sample period: 各頁標示原年分；本輪查閱2026-11-08。
- Known issues:
    - HTTP403回應不是原文；lin_campaign、g6_hsieh、g6_wu僅網頁閱讀工具全文可讀，直接下載失敗明列。
    - review_observations、g6_reviewed_facts、g6_search_decisions為人工整理紀錄，非完整原頁快照；search.txt為搜尋工具結果保存，非每篇全文。
    - txt為HTML文字擷取；PDF頁面已用macOS sips渲染目視相關列。MOI陳奕蓁政黨與當屆登記不同，未覆寫G4。
- Definition for each variable:
    - manifest的url/status/path/sha256/usable：存取與檔案身份；usable不代表研究事實皆核實。
    - G6 facts的relationship/degree/relative_public_office：親屬關係、親等、公職類型；access與basis保留閱讀方式。
    - search decisions的queries/decision/status/reviewed_urls/limitations：實際搜尋與判定；0僅表示指定搜尋未發現。
- Data path: 下方PINS逐檔指定。
- Role: raw / source capture / reviewed observations（分檔說明）
- Verification status: clean kernel逐檔SHA與JSON格式檢查，未更動原始bytes。
- SHA-256: 下方逐檔輸出。
- Next processing notebook: data/code/prepare_clustering_20261108.ipynb


In [1]:
from pathlib import Path
import hashlib,json,pandas as pd
from IPython.display import display
ROOT=next(x for x in [Path.cwd(),*Path.cwd().parents] if (x/'AGENTS.md').is_file() and (x/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()


## Change for Your Own Data
精確檔名與指紋如下。

In [2]:
from pathlib import Path
import hashlib,json,pandas as pd
from IPython.display import display
ROOT=next(x for x in [Path.cwd(),*Path.cwd().parents] if (x/'AGENTS.md').is_file() and (x/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()

PINS={'data/raw/identity_recheck_20261108/chen_bulletin.pdf': '34b52e9ad66d128a4f64205206d9ae05b3ab52928830330d1048814400c591d9', 'data/raw/identity_recheck_20261108/chen_maolin.html': 'd000e734e3f8143122c75a7786bb085e917c4e8c12dbbd27e62fec6e567c7b1f', 'data/raw/identity_recheck_20261108/chen_maolin.txt': 'be6feff1ce34abece9fc8710ea7c7ecaccddb13efe94b73a079715e23d0027f7', 'data/raw/identity_recheck_20261108/chen_moi.html': '528c252ef587198531def9debed27dc609ea18d96de43096ff6a9d401628a2ca', 'data/raw/identity_recheck_20261108/chen_moi.txt': '22acf7e573ffc78d6ba71302299070bf8c448bcd2437eb9f34817c51c47359dd', 'data/raw/identity_recheck_20261108/chen_udn.html': 'c26cbc54c331a72bfd12691a9e159545fdd679c26a4a0a60b8444ffdb298557d', 'data/raw/identity_recheck_20261108/chen_udn.txt': 'b055ca0c23f8c8b0094dc8a058343a132ccaa833af1e1c27fefc76a5276e5243', 'data/raw/identity_recheck_20261108/current_source_reviewed_facts.json': 'afe5083ed40d2995b0d7c6907df5e2805321a6bfeacda05bb121f476a77f9ea9', 'data/raw/identity_recheck_20261108/edu_02e0facb3e5f.html': '0dc323fbf6fbe0c96aea29b7461e2755378c6ca4bbaf86b5864803b8de8f1b37', 'data/raw/identity_recheck_20261108/edu_02e0facb3e5f.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_04d34aeeacf6.html': 'e3a476b77f7a8db1758998458349a10f441aaea3b5751501bcad75ca416b241e', 'data/raw/identity_recheck_20261108/edu_04d34aeeacf6.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_07ba48c85e85.html': '97d3e168b78e0f9ecefd8f9a3dbe836ea4b19d01eb09d038da6a0b9235c391e2', 'data/raw/identity_recheck_20261108/edu_07ba48c85e85.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_0cd62bab8d30.html': '8706be0d94b0e7aa125074c48dd9dae5b4bedd214aa11c04566a7190e45a72d0', 'data/raw/identity_recheck_20261108/edu_0cd62bab8d30.txt': '627c34b5633642b962d07d9c4bdabf3e437a40afec3ae75ed338e6b67dc9426a', 'data/raw/identity_recheck_20261108/edu_0f42bc6d5453.html': 'ab71e212f51569075d6e9fdcf73f3c59c6a0dffde69b62be68db65bafca461cc', 'data/raw/identity_recheck_20261108/edu_0f42bc6d5453.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_0fa9533d4406.html': '77c409b31288066486cfb54fc91a0b4990dd4363397247ad515d7bc7ac07adfd', 'data/raw/identity_recheck_20261108/edu_0fa9533d4406.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_12cecd3ebeb1.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_12cecd3ebeb1.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_18820697dd23.html': 'd8ffbbe403b0ee7390b74114c306c278e9b0a90529d05897ce1a299d05d57b08', 'data/raw/identity_recheck_20261108/edu_18820697dd23.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_2aa85add8b63.html': '449df4af2d2be09d0a3d1b1beaaec7a0260cfec439a30e831b53cbb6f1ce46c9', 'data/raw/identity_recheck_20261108/edu_2aa85add8b63.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_2bc6b319e633.html': 'aae12b213bff16b957234878c496893c26c9378c451d26fe67008bbe66e6f0ed', 'data/raw/identity_recheck_20261108/edu_2bc6b319e633.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_2eb4fc26b165.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_2eb4fc26b165.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_3779b8d372c1.html': '8fbc2b736410669167198a35508b3b1f6de52ee742a2db12edeaf0139393c497', 'data/raw/identity_recheck_20261108/edu_3779b8d372c1.txt': '665aa0864acb9c39d067e9d6af0fee96eed21d5c5daeb71c8c55e5f7216d2358', 'data/raw/identity_recheck_20261108/edu_3dbf11175bbc.html': 'd8c7832cdebea6aad2286276443d655c765b7ad28bfb84788c3eefdc80e4f636', 'data/raw/identity_recheck_20261108/edu_3dbf11175bbc.txt': 'e2b6de733c4d8551000c8bd44b90e516afe76aea810e7745676f887cf32f65fc', 'data/raw/identity_recheck_20261108/edu_3f9e9546e799.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_3f9e9546e799.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_3fe405b88e8c.html': '6480d8bb240d0b5749c626c7ac6cbe42779b075a22568bcff0a0cd4c42f65dc6', 'data/raw/identity_recheck_20261108/edu_3fe405b88e8c.txt': 'e8bcd5f4ef9abe39f74e363aecd194756bdc528cad042096c1528cb584305692', 'data/raw/identity_recheck_20261108/edu_4b6358aa813c.html': '3a0a12f32ea6084424f619dd59d12cb56d857b2b24ac3fa162d03d2aff43ba16', 'data/raw/identity_recheck_20261108/edu_4b6358aa813c.txt': '2c2ee452ded643a343cab01b087449bb419d32553c287b8812b6fd461043d75a', 'data/raw/identity_recheck_20261108/edu_50635a884a83.html': '2210c13ce696038c367cf14537d28d2f9eb6632d54f5c44fd0af0de99cbb7e6b', 'data/raw/identity_recheck_20261108/edu_50635a884a83.txt': '12a7a6a2e25a600970d7898899a19869b095e7e66f4854940409d1b6c0c55676', 'data/raw/identity_recheck_20261108/edu_5b0285ce8dc0.html': '59d924be007248e4ddc1d441a5fd45237d1670ef518b00840be33cd5de4a4883', 'data/raw/identity_recheck_20261108/edu_5b0285ce8dc0.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_5bb27d08fa3f.html': '91f512c4682abdd14ee71e6393626586b95fd4cefe5b678c23e8a322efb1531f', 'data/raw/identity_recheck_20261108/edu_5bb27d08fa3f.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_5c0045a61857.html': '71bf4ac515f31d5b6cd1acf530d09e83884b6b6c9ace5f6d7e4540ef66226db7', 'data/raw/identity_recheck_20261108/edu_5c0045a61857.txt': '32d1176e279f6dc2cde1e47ecd0aee0be23aef2a9aa279890d0cf3557f207af9', 'data/raw/identity_recheck_20261108/edu_5ec760d5cb86.html': '0ae1a6c3c7700d5ca7cb56eef45bbe43eeb6065496c7c1926bc60c4f8c708ad1', 'data/raw/identity_recheck_20261108/edu_5ec760d5cb86.txt': '63f5917ca3d49e98f9fcb3dbdff8b24495590e1d05ad0607cb5162bed009c626', 'data/raw/identity_recheck_20261108/edu_6088f01cbb63.html': '2b24023cc8daeb7f6fe2ccb006cbdd56363341e7603bfbc4f14018f2db01f236', 'data/raw/identity_recheck_20261108/edu_6088f01cbb63.txt': 'd5aed18e71c704fcf905b7c6dd574018b7dd119f3db50294884a60598928595f', 'data/raw/identity_recheck_20261108/edu_681b86a381c9.html': '609d334c53e539485215e2c02b9de027a540d2e3a326e889f92c46cecaa66b74', 'data/raw/identity_recheck_20261108/edu_681b86a381c9.txt': '014760a31e0e33eaa0e1a8f6f564809d2d3815b5cfc6e4e7b807b7c1494c23a1', 'data/raw/identity_recheck_20261108/edu_73e37037c5fc.html': '586007dcfcfb07f7879fc3bdf7adf9723529f928155a0355628051cc919815e2', 'data/raw/identity_recheck_20261108/edu_73e37037c5fc.txt': 'f39584005d363f1d0f65b941b64a39d04b57e9d43f4a10d762cac956242658e9', 'data/raw/identity_recheck_20261108/edu_743bec441d61.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_743bec441d61.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_821d9d4d33e0.html': 'c58ece4569bec5616b5aa95c3172c03a7bc570ac8f7333ca14da4731a7a5a13d', 'data/raw/identity_recheck_20261108/edu_821d9d4d33e0.txt': '189f4e0926d20057fa8793cc967e46a1dc9899fd8b21b5dd64e6bbf09e920d5d', 'data/raw/identity_recheck_20261108/edu_8828b2c22096.html': 'd0f0fb69cade5d9fab5ee5baeee3ff120dab60837bc5d80bf5833e9386b599dd', 'data/raw/identity_recheck_20261108/edu_8828b2c22096.txt': '36c448d4dd113a44192eef0029e0c0bda25c570121991120575ddea7fc33de16', 'data/raw/identity_recheck_20261108/edu_8a931f7bc6ba.html': '7f108ba3d2cc3a5804ce85dda38b9f9fb61a6a397e5dc45e2a63066fe2b72477', 'data/raw/identity_recheck_20261108/edu_8a931f7bc6ba.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_904051aa975f.html': 'a9fe5a6e29183e4eebd8fc2b705fbfdbf97e8563b276d64bbd9cd668ca32c3dd', 'data/raw/identity_recheck_20261108/edu_904051aa975f.txt': '70fc76941538593291f78e5a1243ec4d7d905bd0256f916d7490bd8000bf6b88', 'data/raw/identity_recheck_20261108/edu_9772611b4f6d.html': '95dd66e210240f219649df8ec78c2604a5debf76991b602ddde1e31428e809fa', 'data/raw/identity_recheck_20261108/edu_9772611b4f6d.txt': 'f9ceb5d69a8b8b01b36f9bb7d52ef8f7d1afe31a40888d3db22b1bf6072b0509', 'data/raw/identity_recheck_20261108/edu_9a4847c4a691.html': 'f44df2bed0172023ddbada6cc4a7a52b37af0f720903b97d42039a42334c74f6', 'data/raw/identity_recheck_20261108/edu_9a4847c4a691.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_a4cc4bf704e9.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/edu_a4cc4bf704e9.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/edu_aee3c94631f3.html': '5c6100eb2a358db342b8bde30a2a681bd156d011d5b8057f3227c7e4a4a0df23', 'data/raw/identity_recheck_20261108/edu_aee3c94631f3.txt': 'a1dd620d45a92c843b87b95eb4d863e60f568eda9bf745ed2f9d6aeec06f445b', 'data/raw/identity_recheck_20261108/edu_b4920d182156.html': '08dc6e1d13c1bd364ee788fddbe0654dfc0fbf564c4f302402c493718abdb59e', 'data/raw/identity_recheck_20261108/edu_b4920d182156.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_ba5741600503.html': 'fc4fbc73e10b4f62c08c211fcb81166900772abf271c0cac2e146da77c547d5e', 'data/raw/identity_recheck_20261108/edu_ba5741600503.txt': '9cbeb35a514f424bcfc58ce980f53e79db6890b98b5da8720510c1990bc0b215', 'data/raw/identity_recheck_20261108/edu_c61bfdd11e0e.html': '6dae9b69e03ba24490c1b2503002949bbcee6b3bf6a34032ee24802625195de4', 'data/raw/identity_recheck_20261108/edu_c61bfdd11e0e.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_c8949cc45fbd.html': '63b66367119e5ff676c59ee6ae9e9ec7be4b914baa5bdcd6a33506029f15d039', 'data/raw/identity_recheck_20261108/edu_c8949cc45fbd.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_d1d48715956b.html': 'a6960c3e0e441587a6b3646257e960c917f276cb5fc37772ad901a6cdba8cf7c', 'data/raw/identity_recheck_20261108/edu_d1d48715956b.txt': 'd88c1e30ceb5135a227d4481e4770307a3577dff64c7fccc1c5715ef2e6a9365', 'data/raw/identity_recheck_20261108/edu_d9b8f363d586.html': 'ee3ea390f1c83277b4e5007b61c1fe4aaeaff879077969e81c92b731fccfc168', 'data/raw/identity_recheck_20261108/edu_d9b8f363d586.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_dc5bbe3a0bcf.html': 'a24c6bb65b64ec2c6d6958e6c67cf8140f48da93c2a6e25e5859b0d70b761c57', 'data/raw/identity_recheck_20261108/edu_dc5bbe3a0bcf.txt': '5754f3d5e7337ba29c64376d1856c266114b0641c136449ebe134253597d5506', 'data/raw/identity_recheck_20261108/edu_e01e7795c798.html': '9b8196bab7a2cbb8316fd5a63efcd22cd756c19ff3cf1b5b62fcfa4df9538659', 'data/raw/identity_recheck_20261108/edu_e01e7795c798.txt': '249f34e42cdd4022f45f4c32643ed4e657188dbfda4778376dc822bde6b023b4', 'data/raw/identity_recheck_20261108/edu_e25294c9c53c.html': '57fe4315ee5ddbfdd31b6fcf8559aac8a613d9258edc3e0b3b37258ef788ef3b', 'data/raw/identity_recheck_20261108/edu_e25294c9c53c.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/edu_ed5d2b937362.html': 'a3ff2102565a9f4a1e9089df328fef52b107e88e1b04456dda2fd9fd48904a7a', 'data/raw/identity_recheck_20261108/edu_ed5d2b937362.txt': 'ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f1e1fbab7804a78829b', 'data/raw/identity_recheck_20261108/education_current_leads.json': '42f27f5f244224b4e3a92f04675a327f1c766baca90b9a526a607fbf613a9a27', 'data/raw/identity_recheck_20261108/education_current_manifest.json': '4de3c77a1959e496f370bf231ae2fb5e8a9d5a3dc31c15a525123ff5c148b417', 'data/raw/identity_recheck_20261108/g6_chen.html': '23501da9b9642ddb9c4d06e2aa6c7b68d36d99aba000e336df195ba0a06175e9', 'data/raw/identity_recheck_20261108/g6_chen.txt': '4355d90d918a93df76adc34c374fad253cbb68aeccbc6263ebb29e11898f81ef', 'data/raw/identity_recheck_20261108/g6_hsieh.html': '27895eba249cfa42aaad4d1ad9214501d15f278114fa73f168c507729ae66fb5', 'data/raw/identity_recheck_20261108/g6_hsieh.txt': 'fda3aa5954d255c13978aea765b7f8e07004820ae276a8da72dfd4e7ce688b90', 'data/raw/identity_recheck_20261108/g6_huang.html': '7212c76b2d0029ccebbf5025f14c2dfc6480de72a38bde3a2e22adbcc9667c22', 'data/raw/identity_recheck_20261108/g6_huang.txt': '9a66d73a6377422299b8177161c1e52fe5c6878a68f1c7c0c682135a1d026ca2', 'data/raw/identity_recheck_20261108/g6_initial_search.txt': '92616a5b426d3363f37addbc7e05c8a15c5c249bd25c23b62196796b2b0dc6d3', 'data/raw/identity_recheck_20261108/g6_manifest.json': '389c1e9fd16101b69c4034f96ec185e3c1f2cd01beb89bf4181c9a676a97b764', 'data/raw/identity_recheck_20261108/g6_negative_search.txt': '352f18c47590c89b363cb04859111f6ec7ac75af325068e8503d2afe58edee52', 'data/raw/identity_recheck_20261108/g6_reviewed_facts.json': '8daf2b5722b547ecbb29ef513c739772aea832c10c929c966fb8536e0516ea42', 'data/raw/identity_recheck_20261108/g6_search_decisions.json': 'c5c340d222b858bd84beb086c06788def74003b00d03f191623a38a140466ad8', 'data/raw/identity_recheck_20261108/g6_wu.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/identity_recheck_20261108/g6_wu.txt': '1007c9abee996e66d92446f7b9a92804b6171ed15187fff4a0ea41091c98dc9f', 'data/raw/identity_recheck_20261108/lin_bulletin.pdf': '315fc8e03910f7a6c3fcde973e2539e06949e55bf6419ecdcfca27781cc091fc', 'data/raw/identity_recheck_20261108/lin_campaign.html': '49491740371f299ea945eca8b12c07f7ed173bdcad63867a1228bfdce183bb34', 'data/raw/identity_recheck_20261108/manifest.json': '0ff7157d45a2c22014f7892ab6e8e1e4891636d24874543eb3aaa580138f0d21', 'data/raw/identity_recheck_20261108/review_observations.json': '34f8284ac91178655eb6bc2e95633d8c6f67e4d3f2bff31024a0731c38e65458'}
rows=[]
for f,expected in PINS.items():
 q=ROOT/f;assert sha(q)==expected
 if q.suffix=='.json':json.loads(q.read_text())
 rows.append(dict(path=f,bytes=q.stat().st_size,sha256=expected))
df=pd.DataFrame(rows)


## Summary
### Sample

In [3]:
display(df.head(10))

,path,bytes,sha256
0,data/raw/identity_recheck_20261108/chen_bullet...,2850173,34b52e9ad66d128a4f64205206d9ae05b3ab5292883033...
1,data/raw/identity_recheck_20261108/chen_maolin...,172946,d000e734e3f8143122c75a7786bb085e917c4e8c12dbbd...
2,data/raw/identity_recheck_20261108/chen_maolin...,7276,be6feff1ce34abece9fc8710ea7c7ecaccddb13efe94b7...
3,data/raw/identity_recheck_20261108/chen_moi.html,92190,528c252ef587198531def9debed27dc609ea18d96de430...
4,data/raw/identity_recheck_20261108/chen_moi.txt,5244,22acf7e573ffc78d6ba71302299070bf8c448bcd2437eb...
5,data/raw/identity_recheck_20261108/chen_udn.html,53273,c26cbc54c331a72bfd12691a9e159545fdd679c26a4a0a...
6,data/raw/identity_recheck_20261108/chen_udn.txt,3373,b055ca0c23f8c8b0094dc8a058343a132ccaa833af1e1c...
7,data/raw/identity_recheck_20261108/current_sou...,11944,afe5083ed40d2995b0d7c6907df5e2805321a6bfeacda0...
8,data/raw/identity_recheck_20261108/edu_02e0fac...,5593,0dc323fbf6fbe0c96aea29b7461e2755378c6ca4bbaf86...
9,data/raw/identity_recheck_20261108/edu_02e0fac...,58,ae0b26c68939e8214056c2517af3b5091f1c674c8aef5f...


### Summary Stats

In [4]:
display(df[['bytes']].describe());print('verified files',len(df));print(json.dumps(json.loads((ROOT/'data/raw/identity_recheck_20261108/g6_search_decisions.json').read_text()),ensure_ascii=False,indent=2))

,bytes
count,1.130000e+02
mean,1.078753e+05
std,4.744350e+05
min,1.700000e+01
25%,1.340000e+02
50%,4.546000e+03
75%,2.442400e+04
max,3.454100e+06


verified files 113
[
  {
    "candidate_name": "蘇輝湟",
    "queries": [
      "\"蘇輝湟\" 家族 父親 配偶",
      "\"蘇輝湟\" 政治家族 父母 兄弟 妻子"
    ],
    "decision": "0",
    "status": "none_found_online_screen",
    "checked_date": "2026-11-08",
    "reviewed_urls": [
      "https://whoareyou.readr.tw/person/44342"
    ],
    "limitations": "搜尋結果未發現符合三親等／配偶且有公職的親屬；READr人物頁全文未列親屬。Yahoo相關新聞404，列存取失敗，不當作已讀全文。依使用者明確規則編0，並非已證明無親屬。"
  },
  {
    "candidate_name": "林志成",
    "queries": [
      "\"林志成\" 台北市長 家族 父親 配偶",
      "\"林志成\" \"健美\" 政治家族 父母 妻子"
    ],
    "decision": "0",
    "status": "none_found_online_screen",
    "checked_date": "2026-11-08",
    "reviewed_urls": [
      "https://ctbbf2010.wixsite.com/ctbbf/關於我們"
    ],
    "limitations": "健美背景脈絡搜尋未發現符合範圍的公職親屬；排除80歲大溪音樂老師同名者。協會全文僅核對職涯，不證明無親屬。依使用者規則編0。"
  },
  {
    "candidate_name": "郭璽",
    "queries": [
      "\"郭璽\" \"政治世家\"",
      "\"郭璽\" \"林黎真\" 議員 公職 家族"
    ],
    "decision": "",
    "status": "family_leads_need_full_text_review",
    "ch